**Data sets per year for age group 16-29 years old (including 16 and 29)**

In [1]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 743.7/743.7 kB 10.3 MB/s  0:00:00


In [2]:
import pyreadstat
import polars as pl
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [4]:
## Create a new dataframe "YYYY_AG1629" from ...pers... and fill with filtered list, filter set to alterx=16-29

## for 2001 - 2022, the age groups are:
#   ...
#   7 = 7 bis 9 Jahre
#   10-74 = Jahre
#   75 = 75 bis 79 Jahre
#   ...

## for 1991 the age groups are:
# ...
# 4 = 12-15
# 5 = 15-20
# 6 = 20-25
# 7 = 25-30
# 8 = 30-35
# ...

## 1991
df_1991_pandas, metadata = pyreadstat.read_por("1991/zbe_ps.por")
df_1991 = pl.from_pandas(df_1991_pandas)
df_1991_AG1629 = df_1991.filter(
    pl.col('E51X').is_between(5,7))

## 2001
df_2001 = pl.read_csv('2001/zbe01_02puf_p.csv', separator=",", ignore_errors=True)
df_2001_AG1629 = df_2001.filter(
    pl.col('PH01B2X').is_between(16,29))

## 2012
df_2012 = pl.read_csv('2012/zve13_puf_pers.csv', separator=";", ignore_errors=True)
df_2012_AG1629 = df_2012.filter(
    pl.col('alterx').is_between(16,29))

## 2022
df_2022 = pl.read_csv('2022/zve_pers_puf_2022_v1.csv', separator=";", ignore_errors=True)
df_2022_AG1629 = df_2022.filter(
    pl.col('alterx').is_between(16,29))

In [5]:
## Create a new dataframe "YYYY_AG16-29_activites" from ...sum... with only rows according to listed id_persx of "YYYY_AG16-29"

## 1991
pers_ids_1991_AG1629 = df_1991_AG1629.get_column('ID1').unique()

df_1991_activities_pandas, metadata = pyreadstat.read_por("1991/zbe_sum.por")
df_1991_activities = pl.from_pandas(df_1991_activities_pandas)

df_1991_AG1629_activities = df_1991_activities.filter(
    pl.col('ID1').is_in(pers_ids_1991_AG1629.implode()))

## 2001
pers_ids_2001_AG1629 = df_2001_AG1629.get_column('IDPERS').unique()

df_2001_activities = pl.read_csv('2001/zbe01_02puf_sum.csv', separator=",", ignore_errors=True)

df_2001_AG1629_activities = df_2001_activities.filter(
    pl.col('IDPERS').is_in(pers_ids_2001_AG1629.implode()))

## 2012
pers_ids_2012_AG1629 = df_2012_AG1629.get_column('id_persx').unique()

df_2012_activities = pl.read_csv('2012/zve13_puf_sum.csv', separator=";", ignore_errors=True)

df_2012_AG1629_activities = df_2012_activities.filter(
    pl.col('id_persx').is_in(pers_ids_2012_AG1629.implode()))

## 2022
pers_ids_2022_AG1629 = df_2022_AG1629.get_column('id_persx').unique()

df_2022_activities = pl.read_csv('2022/zve_sum_puf_2022_v1.csv', separator=";", ignore_errors=True)

df_2022_AG1629_activities = df_2022_activities.filter(
    pl.col('id_persx').is_in(pers_ids_2022_AG1629.implode()))

In [68]:
## Create a unique list of activity codes
## Replace the activity codes per YYYY with the unified codes

## Columns relevant for the activities
## 1991: ZH... (in minutes)
## 2001: ZH... (in minutes)
## 2012: ha_... (in minutes)
## 2022: ha_... (in minutes)

## Rename ZH to ha
def rename_activity_column(column):
    if column.startswith("ZH"):
        return f"ha_{column[2:]}"
    return column

df_1991_AG1629_activities = df_1991_AG1629_activities.rename(rename_activity_column)
df_2001_AG1629_activities = df_2001_AG1629_activities.rename(rename_activity_column)

## Trim the dataframes to only the activity columns
df_1991_AG1629_activities = df_1991_AG1629_activities.select(
    pl.col(r'^ha_.*$')
).select(
    pl.exclude('ha_OEFF','ha_FAM','ha_PERS')
)

df_2001_AG1629_activities = df_2001_AG1629_activities.select(
    pl.col(r'^ha_.*$')
)

df_2012_AG1629_activities = df_2012_AG1629_activities.select(
    pl.col(r'^ha_.*$')
)

df_2022_AG1629_activities = df_2022_AG1629_activities.select(
    pl.col(r'^ha_.*$')
)

df_2022_AG1629_activities.sample(10)

ha_1,ha_2,ha_3,ha_4,ha_5,ha_6,ha_7,ha_8,ha_9,ha_11,ha_12,ha_13,ha_21,ha_22,ha_23,ha_24,ha_31,ha_32,ha_33,ha_34,ha_35,ha_36,ha_41,ha_42,ha_43,ha_44,ha_45,ha_46,ha_47,ha_48,ha_49,ha_51,ha_52,ha_53,ha_61,ha_62,ha_63,…,ha_812,ha_813,ha_814,ha_815,ha_819,ha_820,ha_830,ha_841,ha_842,ha_849,ha_921,ha_922,ha_923,ha_929,ha_931,ha_934,ha_939,ha_941,ha_945,ha_946,ha_947,ha_948,ha_949,ha_951,ha_952,ha_953,ha_959,ha_961,ha_962,ha_969,ha_970,ha_980,ha_991,ha_992,ha_997,ha_998,ha_999
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""13:40""","""0:00""","""0:00""","""2:30""","""0:00""","""4:50""","""1:00""","""0:00""","""2:00""","""10:20""","""2:30""","""0:50""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:20""","""1:20""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:50""","""0:00""","""0:00""","""0:00""","""0:00""","""3:30""","""1:20""",…,"""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""1:30""","""0:30""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00"""
"""10:30""","""0:00""","""0:00""","""2:30""","""0:00""","""4:00""","""0:00""","""6:40""","""0:20""","""8:40""","""1:10""","""0:40""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""2:30""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""2:20""","""0:00""","""1:40""",…,"""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""6:10""","""0:30""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:20""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00"""
"""11:00""","""0:00""","""7:00""","""0:00""","""0:00""","""0:00""","""0:00""","""3:50""","""2:10""","""9:40""","""1:00""","""0:20""","""0:00""","""0:00""","""0:00""","""0:00""","""5:10""","""0:00""","""0:00""","""0:00""","""1:50""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""",…,"""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""3:50""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""2:10""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00"""
"""12:00""","""0:00""","""0:00""","""6:30""","""0:00""","""1:10""","""0:00""","""2:20""","""2:00""","""9:30""","""1:00""","""1:30""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""1:30""","""1:10""","""0:00""","""0:00""","""0:00""","""1:00""","""2:50""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:10""","""1:00""","""0:00""",…,"""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""2:20""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""1:00""","""0:40""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:00""","""0:20""","""0:00""","""0:00""","""0:00""","""0:00""","""

In [ ]:
## Harmonise the time values

# 1991 uses minutes (f64), 2001 uses minutes (i64), 2012 uses seconds (i64), 2022 uses hh:mm (str)

In [ ]:
## Create a new dataframe "df_YYYY_AG1629_activites_means" with the activity codes as rows and a column for YYYY
## for each column in df_activities: create the mean and write to the row of the activity code and the column YYYY

# Create a nested list with pairs of column and mean of column: [[ha_1, mean(ha_1)], [...]]
def column_and_mean(dataframe):
    
    rows = []
    
    for column in dataframe.columns:
        column_mean = dataframe.get_column(column).mean()
        rows.append([column, column_mean])
    
    return rows

## 1991
df_1991_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2012_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2001
df_2001_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2001_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2012
df_2012_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2012_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2022
df_2022_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2022_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

activity code,mean
str,null
"""ha_12""",null
"""ha_84""",null
"""ha_43""",null
"""ha_934""",null
"""ha_21""",null
"""ha_523""",null
"""ha_632""",null
"""ha_411""",null
"""ha_64""",null


In [50]:
## MAP 1991 activity codes to 2001 activity codes

# defining a function to map old names to new names per row entry

def map_rows(df_map, df_data):
    map = dict(df_map.iter_rows())
    activity_code = df_data.columns[0]

    return df_data.with_columns(
        pl.col(activity_code).replace(map)
    )

# 1991 -> 2001 Mapped new codes manually in https://docs.google.com/spreadsheets/d/1_sxoe-Q-1655vzxUeIR0q9Kb5vrXVbpMbM02MPW90so/edit?gid=406327516#gid=406327516
df_mapping_1991_2001 = pl.read_csv('1991/mapping_1991.csv', separator=',')

df_1991_AG1629_activities_means = map_rows(df_mapping_1991_2001, df_1991_AG1629_activities_means)

## Map 1991 and 2001 activity code to match 2012/2022 activity codes with 
df_mapping_2001_2012 = pl.read_excel('2012/zve_2012-2013_puf_kl-akt-erl.xlsx', engine='calamine', sheet_name='ZBE 2002 --> ZVE 2013')

# Add new codes to 1991 & 2001 dataset with 0 for means
new_codes = []
for rows in df_mapping_2001_2012.filter(
    pl.col('ZBE 01/02') == "-"
)['ZVE 12/13']:
    new_codes.append(f"ha_{int(rows)}")

df_new_codes = pl.DataFrame({
    'activity code': new_codes,
    'mean': 0.0
})

df_1991_AG1629_activities_means = pl.concat([df_1991_AG1629_activities_means, df_new_codes])
df_2001_AG1629_activities_means = pl.concat([df_2001_AG1629_activities_means, df_new_codes])

# Drop new codes from df_mapping_2001_2012
df_mapping_2001_2012 = df_mapping_2001_2012.filter(
    pl.col('ZBE 01/02') != "-"
)

# Drop leading 0s from df_mapping_2001_2012
df_mapping_2001_2012 = df_mapping_2001_2012.with_columns(pl.col('ZBE 01/02').cast(pl.Int64))

# Change i64 to "ha_" + i64
df_mapping_2001_2012 = df_mapping_2001_2012.with_columns(
    ("ha_" + pl.col('ZBE 01/02').cast(pl.String)).alias("Old"),
    ("ha_" + pl.col('ZVE 12/13').cast(pl.String)).alias("New"),
)
# Drop non relevant columns
df_mapping_2001_2012 = df_mapping_2001_2012.drop(pl.col('ZBE 01/02', 'ZVE 12/13', 'Label 01/02', 'Label 12/13', '__UNNAMED__4'))

# Map 1991 and 2001 datasets
df_1991_AG1629_activities_means = map_rows(df_mapping_2001_2012, df_1991_AG1629_activities_means)
df_2001_AG1629_activities_means = map_rows(df_mapping_2001_2012, df_2001_AG1629_activities_means)

# merging rows with the same new code, sum of means
df_1991_AG1629_activities_means = df_1991_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())
df_2001_AG1629_activities_means = df_2001_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())

Could not determine dtype for column 4, falling back to string
/var/folders/5_/0wmfbw9s5y34jq82848gv2qw0000gn/T/ipykernel_6551/1435002590.py:19: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df_mapping_2001_2012 = pl.read_excel('2012/zve_2012-2013_puf_kl-akt-erl.xlsx', engine='calamine', sheet_name='ZBE 2002 --> ZVE 2013')


**Data set for yearly comparison**

In [52]:
## Create a new dataframe "df_Year-by-Year_AG1629_overview" with the activity codes as rows and 1991, 2001, 2012 and 2022 as columns
## Add all data points of the years from dataframes "df_YYYY_AG1629_activites" to "df_Year-by-Year_AG1629_overview"

## Create rows with the activity code, find the row in 1991, 2001, 2012 and 2022 and add their means to the row. Return the rows.

def merge(df1, df2, df3, df4):
    
    rows = []
    
    for row1, row2, row3, row4 in zip(df1.iter_rows(), df2.iter_rows(), df3.iter_rows(), df4.iter_rows()):
        
        rows.append(
            [
                row1[0],
                row1[1],
                row2[1],
                row3[1],
                row4[1]
            ]
        )
    
    return rows

## merge all dataframes

df_YBY_AG1629_means = pl.DataFrame(
    merge(
        df_1991_AG1629_activities_means,
        df_2001_AG1629_activities_means,
        df_2012_AG1629_activities_means,
        df_2022_AG1629_activities_means,
    ),
    schema=["activity code", "1991", "2001", "2012", "2022"],
    orient="row",
)

df_YBY_AG1629_means.sample(10)

activity code,1991,2001,2012,2022
str,f64,f64,f64,null
"""ha_452""",79.429735,6.369257,410.183299,null
"""ha_842""",867.209776,56.663121,807.331976,null
"""ha_999""",78.615071,0.0,124.507807,null
"""ha_466""",136.320434,1.071737,79.565513,null
"""ha_761""",329.531568,43.279037,2811.812627,null
"""ha_475""",55.940258,21.074729,239.10387,null
"""ha_991""",601.086219,2.764776,889.884589,null
"""ha_464""",160.624576,0.0,30.685675,null
"""ha_469""",136.456212,1.88651,801.765105,null


In [ ]:
## Add trends to compare year-over-year and a trend for 1991-2022 for each activity code for "df_Year-by-Year_AG1629_overview"



In [ ]:
## Write dataframe "df_Year-by-Year_AG1629_overview" to "Year-by-Year_AG1629_overview.csv"